In [ ]:
import Pkg
Pkg.activate("..")
Pkg.instantiate()

using CSV
using DataFrames
using Plots

In [ ]:
dir = "../data/results/exp5_scalability"
df = CSV.read(joinpath(dir, "exp5_scalability.csv"), DataFrame)

function linear_fit(x, y)
    x̄, ȳ = sum(x) / length(x), sum(y) / length(y)
    a = sum((x .- x̄) .* (y .- ȳ)) / sum((x .- x̄) .^ 2)
    b = ȳ - a * x̄
    r2 = 1 - sum((y .- (a .* x .+ b)) .^ 2) / sum((y .- ȳ) .^ 2)
    return a, b, r2
end

for dataset in unique(df.Dataset)
    data = filter(r -> r.Dataset == dataset, df)
    n = data.Transactions ./ 1000
    a, b, r2 = linear_fit(n, data.Base_Time_s)

    p = plot(n, data.Base_Time_s, label = "Basic implementation", marker = :square, lw = 2)
    plot!(p, n, data.Opt_Time_s, label = "Optimized implementation", marker = :diamond, lw = 2)
    all(isnan, data.SPMF_Time_s) || plot!(p, n, data.SPMF_Time_s, label = "SPMF", marker = :circle, lw = 2)
    plot!(p, n, data.Base_Build_s, label = "Basic: FP-tree construction only", marker = :utriangle, ls = :dot)
    plot!(p, n, a .* n .+ b, label = "Linear fit (R² = $(round(r2, digits = 3)))", ls = :dash, color = :gray)
    plot!(p, title = "Scalability ($(uppercasefirst(dataset)), minsup = $(data.Minsup[1]))",
          xlabel = "Number of transactions (K)", ylabel = "Runtime (Seconds)", legend = :topleft, grid = true)
    display(p)
    savefig(p, joinpath(dir, "plot_scalability_$(dataset).png"))

    # Thời gian trên mỗi 1000 giao dịch: gần như nằm ngang => tuyến tính
    q = plot(n, data.Base_Time_s ./ n, label = "Basic", marker = :square, lw = 2)
    plot!(q, n, data.Opt_Time_s ./ n, label = "Optimized", marker = :diamond, lw = 2)
    plot!(q, title = "Runtime per 1K transactions ($(uppercasefirst(dataset)))",
          xlabel = "Number of transactions (K)", ylabel = "Seconds / 1K transactions", ylims = (0, Inf), grid = true)
    display(q)
    savefig(q, joinpath(dir, "plot_time_per_trans_$(dataset).png"))
end